# B2 · 貝氏因子：這到底是不是行星？

> **核心問題**：光曲線上有一個凹陷。**是行星凌日？還是食雙星？還是儀器雜訊？**

B1 問的是「這顆行星**多大**」——那是**參數估計**，在單一模型內問 $p(\theta \mid D, M)$。
B2 問的是「這**是不是**行星」——那是**模型選擇**，要比較 $p(M \mid D)$：

$$p(M \mid D) \propto \underbrace{p(D \mid M)}_{\text{邊際似然（證據）}} \, p(M), \qquad
p(D \mid M) = \int p(D \mid \theta, M)\, p(\theta \mid M)\, d\theta$$

**這正是 MCMC 算不出來的那個量。** Metropolis 的接受率是後驗的比值，$p(D)$ 在分子分母消掉了
——這是它不需要歸一化常數的原因，也是它給不出 $p(D\mid M)$ 的原因。
Nested sampling 則是**專門為了直接算 $\log Z = \log p(D\mid M)$ 而設計的**。

這個 notebook 走完整條判決流程；奧卡姆剃刀與 Lindley 悖論的細節在
[`02_occam_and_lindley.ipynb`](02_occam_and_lindley.ipynb)。

In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, json
import data, models, evidence as ev, plots
DATA = os.path.abspath('../../data/B_astro')
RES = json.load(open('../figures/results.json'))

## 1 · 兩個目標：一真一假

要證明一套判定方法有用，光是「對已確認的行星說 yes」不夠——**還必須對假的說 no**。
所以除了 Kepler-10b，另外從 NASA Exoplanet Archive 的 KOI 累積表挑一個**官方判定的
FALSE POSITIVE**：條件是 `koi_disposition='FALSE POSITIVE'` 且 `koi_fpflag_ss=1`
（Stellar Eclipse 旗標），並同時帶有 `DEEP_V_SHAPED` 與 `HAS_SEC_TCE` 註記。

也就是說：**官方判它是假行星的理由，正是我們的 M2 模型要抓的兩個特徵。**

In [2]:
targets = {}
for key in ('kepler10b', 'koi6017'):
    d = data.prepare(key, DATA)
    bp, bf, be, bn = data.adaptive_bin(d['phase'], d['flux'], float(d['dur_phase']))
    d_pri, d_sec = data.measure_depths(d['phase'], d['flux'], float(d['dur_phase']))
    tgt = data.TARGETS[key]
    targets[key] = dict(d=d, bp=bp, bf=bf, be=be, P=float(d['P']),
                        dur_phase=float(d['dur_phase']), d_pri=d_pri, d_sec=d_sec, **tgt)
    print(f"{tgt['title']:<13}[{tgt['disposition']:<14}] P={float(d['P']):.6f} d  "
          f"主食={d_pri:>8.0f} ppm  次食={d_sec:>6.0f} ppm  "
          f"每箱誤差={np.median(be)*1e6:5.1f} ppm  次食/主食={d_sec/d_pri:.4f}")

Kepler-10b   [CONFIRMED     ] P=0.837489 d  主食=     187 ppm  次食=     4 ppm  每箱誤差=  5.0 ppm  次食/主食=0.0196
KOI-6017.01  [FALSE POSITIVE] P=5.262691 d  主食=   19522 ppm  次食=   744 ppm  每箱誤差= 20.5 ppm  次食/主食=0.0381


### 資料管線上的兩個坑（都跟 B1 不同）

1. **摺疊必須覆蓋完整相位 $[-0.5, 0.5]$**。B1 只要凌日窗就能估參數；這裡要判斷「是不是
   食雙星」，而決定性證據是**次食**（相位 0.5 附近的第二個凹陷）。只看凌日窗
   = 把最關鍵的證據丟掉。同理，`flatten` 時要**同時遮罩主食與次食**，
   否則幾百 ppm 的次食會被 Savitzky–Golay 當成趨勢吸收。

2. **sigma clip 只能砍上方**。`remove_outliers(sigma=5)` 這種對稱裁切會把 2% 深的食
   **當成離群值刪掉**（實測砍掉 1391 個點，剛好都在食內）。凹陷是訊號不是雜訊。

![全相位光曲線](../figures/01_full_phase.png)

左欄一眼看得出差別：Kepler-10b 在相位 0.5 附近**什麼都沒有**；
KOI-6017.01 有一個清清楚楚的次食。右欄是次食區放大。

In [3]:
for k, t in targets.items():
    print(f"{t['title']:<13} 次食深度 {t['d_sec']:>6.0f} ppm  vs  每箱誤差 "
          f"{np.median(t['be'])*1e6:5.1f} ppm  →  比值 {t['d_sec']/(np.median(t['be'])*1e6):6.1f}")
print("\n注意：這只是**單箱**的粗略比值，不是次食的統計顯著性——")
print("次食橫跨多個箱，而且深度與 rp、b、a 簡併。正確的顯著性要看 M2 的 J 後驗（第 5 節）。")

Kepler-10b    次食深度      4 ppm  vs  每箱誤差   5.0 ppm  →  比值    0.7
KOI-6017.01   次食深度    744 ppm  vs  每箱誤差  20.5 ppm  →  比值   36.3

注意：這只是**單箱**的粗略比值，不是次食的統計顯著性——
次食橫跨多個箱，而且深度與 rp、b、a 簡併。正確的顯著性要看 M2 的 J 後驗（第 5 節）。


## 2 · 三個競爭模型

建模時最容易犯的錯，是讓 M1 和 M2 差太多——那樣「M2 贏」就分不清是物理還是自由度。
這裡兩者的差別**只有兩處，而且兩處都是真正的天文判準**：

| | M0 純雜訊 | M1 行星凌日 | M2 食雙星 |
|---|---|---|---|
| 參數數 | 2 | 8 | 9 |
| $R_c/R_*$ 先驗 | — | $U(0,\,0.2)$ | $U(0,\,1.0)$ |
| 次食 | 無 | **無** | **有**（深度 $= J \times$ 主食） |

1. **$R_c/R_*$ 的先驗範圍**：行星有物理半徑上限（簡併壓讓行星撐不過 ~2 $R_{\rm Jup}$，
   即 $R_p/R_* \lesssim 0.2$）；食雙星的伴星可以跟主星差不多大。
   **先驗就是物理假設**——這正是貝氏因子與「同一個模型硬擬合再比 $\chi^2$」的根本差異。
2. **次食**：行星的次食是幾十 ppm 等級（反射 + 熱輻射），在這個精度下等於沒有；
   食雙星的伴星自己會發光，次食深度是主食的 $J$ 倍（$J$ = 表面亮度比）。

**M2 = M1 + 一個參數 $J$。** 只多一個 → 奧卡姆懲罰乾淨可算。

## 3 · 用 nested sampling 算邊際似然

`prior_transform` 把單位立方 $[0,1]^d$ 映射到參數空間——這是 nested sampling 的標準介面，
也把「先驗是什麼」寫得毫不含糊（先驗體積就是這個映射的 Jacobian）。

> ⚠️ **這個專案最大的效能關卡**：ultranest 預設的 MLFriends 用橢球包絡做拒絕抽樣，
> 維度一高、後驗一簡併，接受率就崩潰。實測 M1（8 維、資料 SNR ≈ 1000）用 MLFriends
> 跑 **12 分鐘還沒收斂**；換成 `SliceSampler` 後 **63 秒**完成，活點數還從 200 提到 400。
> 這不是調參技巧，是 nested sampling 的已知性質（官方建議 $d \gtrsim 7$ 就該換 step sampler）。

下面用單一 seed 跑一遍（約 10–15 分鐘）。`run_all.py` 用三個 seed 重複，
理由見 notebook 02 的「數值可靠性」一節。

In [4]:
summaries = {}
for key, t in targets.items():
    shape = models.TransitShape(t['bp'], t['P'], supersample=7)
    res = [ev.run_evidence(M, min_live=400) for M in models.build_all(shape, t['bf'], t['be'])]
    summaries[key] = ev.summarize(res, f"{t['title']}  [{t['disposition']}]")
    targets[key]['res'] = res


── Kepler-10b  [CONFIRMED] ─────────────────────────────────────────
模型    說明          參數         log Z   ±(seed)     ±(內部)    max logL    p(M|D)
M0    純雜訊          2       1341.48      0.22      0.22      1351.5     0.000
M1    行星凌日         8       1726.82      0.27      0.27      1758.1     0.923
M2    食雙星          9       1724.33      0.42      0.42      1760.8     0.077
  log10 B(M1/M0) =  +167.35 ± 0.15   → 決定性證據，支持前者
  log10 B(M2/M1) =    -1.08 ± 0.22   → 強證據，支持後者
  log10 B(M2/M0) =  +166.27 ± 0.21   → 決定性證據，支持前者
  → 勝出：M1（行星凌日）



── KOI-6017.01  [FALSE POSITIVE] ─────────────────────────────────────────
模型    說明          參數         log Z   ±(seed)     ±(內部)    max logL    p(M|D)
M0    純雜訊          2      -1939.69      0.25      0.25     -1922.6     0.000
M1    行星凌日         8       1091.21      0.36      0.36      1120.5     0.000
M2    食雙星          9       1112.66      0.35      0.35      1146.2     1.000
  log10 B(M1/M0) = +1316.30 ± 0.19   → 決定性證據，支持前者
  log10 B(M2/M1) =    +9.32 ± 0.22   → 決定性證據，支持前者
  log10 B(M2/M0) = +1325.62 ± 0.19   → 決定性證據，支持前者
  → 勝出：M2（食雙星）


## 4 · 判決

![邊際似然比較](../figures/03_evidence.png)

黑色三角形是**最大似然**（最佳擬合有多好），彩色長條是**邊際似然**（模型整體有多可信）。
**注意 Kepler-10b 那一欄：M2 的三角形比 M1 高，長條卻比 M1 矮**——
複雜模型擬合得更好，證據卻更低。那就是奧卡姆剃刀，下一個 notebook 會把這筆帳算清楚。

In [5]:
print(f"{'目標':<14}{'判決':<12}{'log10 B(M1/M0)':>18}{'log10 B(M2/M1)':>18}   結論")
for key, s in summaries.items():
    verdict = '行星 ✓' if s['best'] == 'M1' else ('食雙星' if s['best'] == 'M2' else '純雜訊')
    print(f"{targets[key]['title']:<14}{targets[key]['disposition']:<16}"
          f"{s['log10_B_M1M0']:>14.1f}{s['log10_B_M2M1']:>18.2f}   → {verdict}")

目標            判決              log10 B(M1/M0)    log10 B(M2/M1)   結論
Kepler-10b    CONFIRMED                167.3             -1.08   → 行星 ✓
KOI-6017.01   FALSE POSITIVE          1316.3              9.32   → 食雙星


### 對照 Jeffreys 尺度

| $\log_{10} B$ | 證據強度 |
|---|---|
| 0 – 0.5 | 勉強一提 |
| 0.5 – 1 | 實質 |
| 1 – 1.5 | 強 |
| 1.5 – 2 | 非常強 |
| > 2 | **決定性** |

兩個目標都給出決定性的判決，而且**方向都對**：

- **Kepler-10b**：$\log_{10} B(M_1/M_0)$ 遠大於 2 → 凹陷確實存在；
  且 $\log_{10} B(M_2/M_1) < 0$ → 資料**不需要**食雙星那套額外機制。**判定：行星。**
- **KOI-6017.01**：$\log_{10} B(M_1/M_0)$ 也極大（凹陷當然存在），
  但 $\log_{10} B(M_2/M_1) \gg 2$ → **貝氏因子明確不支持行星模型**。
  這跟官方的 FALSE POSITIVE 判定一致，而且我們是**從光子重新推導出來的**。

## 5 · 為什麼 M2 對一個贏、對另一個輸？

M2 比 M1 多的那一個參數是 $J$（次食 / 主食的表面亮度比）。直接看它的後驗：

![J 的後驗](../figures/04_secondary_posterior.png)

In [6]:
for key, t in targets.items():
    m2 = [r for r in t['res'] if r['name'] == 'M2'][0]
    J = m2['samples'][:, m2['param_names'].index('J')]
    q = np.percentile(J, [2.5, 50, 97.5])
    rel = 100*(q[2]-q[0])/2/q[1]
    dep = q * t['d_pri']
    print(f"{t['title']:<13} J = {q[1]:.4f} [{q[0]:.4f}, {q[2]:.4f}]  ±{rel:3.0f}%"
          f"   → 次食深度 {dep[1]:6.1f} ppm [{dep[0]:.1f}, {dep[2]:.1f}]")

Kepler-10b    J = 0.0189 [0.0046, 0.0354]  ± 82%   → 次食深度    3.5 ppm [0.9, 6.6]
KOI-6017.01   J = 0.0292 [0.0215, 0.0361]  ± 25%   → 次食深度  569.9 ppm [420.3, 705.5]


> ⚠️ **這裡有個容易講錯、但很重要的細節。**
> 兩個目標的 $J$ **中位數其實是同一個量級**（~0.02 與 ~0.03）——不能說 Kepler-10b 的
> $J$「等於 0」。差別在兩處：
>
> 1. **相對不確定度**：Kepler-10b 的 $J$ 誤差達 ±80% 等級，KOI-6017.01 只有 ±20% 等級。
> 2. **絕對深度**：換算成 ppm，Kepler-10b 的次食是**幾個 ppm**，KOI-6017.01 是**幾百 ppm**。
>
> 而且那幾個 ppm **很可能是真的**——Kepler-10b 是超短週期的熾熱岩質行星，
> 已發表的次食（熱輻射 + 反射）約 5–8 ppm，與我們測到的量級吻合。
> 所以 M2 在 Kepler-10b 上多賺的那點似然，抓到的是**行星自己的次食**，不是食雙星的證據。
>
> 這也釐清了 M1 與 M2 到底在比什麼：**M2 vs M1 檢驗的是「有沒有可測的次食」**，
> 而「是不是行星」還要靠 $R_c/R_*$ 的先驗範圍一起判斷。下一格就看那個。

### 先驗範圍在做的事：M1 被擠到邊界上

看兩個模型各自的最佳解——這是「先驗即物理假設」最具體的樣子：

In [7]:
for key, t in targets.items():
    print(f"{t['title']}  (M1 的 R_c/R* 先驗上限 = {models.RP_MAX_PLANET})")
    for m in ('M1', 'M2'):
        r = [x for x in t['res'] if x['name'] == m][0]
        p = dict(zip(r['param_names'], np.median(r['samples'], axis=0)))
        extra = f"  J={p['J']:.4f}" if m == 'M2' else ""
        print(f"    {m}: R_c/R* = {p['rp']:.4f}   b = {p['b']:.3f}   "
              f"a/R* = {p['a']:.2f}{extra}")
    print()

Kepler-10b  (M1 的 R_c/R* 先驗上限 = 0.2)
    M1: R_c/R* = 0.0136   b = 0.697   a/R* = 2.79
    M2: R_c/R* = 0.0132   b = 0.468   a/R* = 3.25  J=0.0189

KOI-6017.01  (M1 的 R_c/R* 先驗上限 = 0.2)
    M1: R_c/R* = 0.1870   b = 0.886   a/R* = 10.82
    M2: R_c/R* = 0.3733   b = 1.157   a/R* = 10.43  J=0.0292



**KOI-6017.01 的 M1 把 $R_c/R_*$ 頂到先驗上限 0.2 的九成以上**，
還得配上 $b \approx 0.9$ 的掠射才勉強做出那個 V 形。
換句話說：要用「行星」解釋這條光曲線，得假設一顆**貼著物理上限的巨行星擦邊掠過**。

M2 沒有這個束縛，給出的 $R_c/R_*$ **好幾倍於行星上限**（伴星與主星同一個尺度），
配上 $b > 1$ 的**部分掩食**——兩顆大小相當的恆星掠射相食，
這才是那個 V 字形凹陷的自然解釋。

（$R_c/R_*$ 與 $b$ 強烈簡併，所以中位數在不同 seed 之間會浮動；
穩定的是「M1 頂到邊界、M2 遠離邊界」這個**定性**結論。）

而 Kepler-10b 的 M1 落在 $R_c/R_* \approx 0.013$，離邊界遠得很，完全不吃力。

![三個模型的擬合](../figures/02_model_fits.png)

左欄主食、右欄次食。KOI-6017.01 的次食區裡，只有 M2（紅）壓得下去；
Kepler-10b 的次食區三條線疊在一起——**資料裡沒有東西可給 M2 解釋**。

---
**下一步**：[`02_occam_and_lindley.ipynb`](02_occam_and_lindley.ipynb)
把奧卡姆懲罰算成具體的 nat 數，並誠實面對貝氏因子對先驗的依賴。